In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import duckdb
file_path = '/content/drive/MyDrive/UM DATATHON/Preliminary Round /swift_umdac_datathon_2026_fictitious_payment_data.csv'
duckdb.sql(f"DESCRIBE SELECT * FROM '{file_path}'").df()[['column_name', 'column_type']]


# 1. JOY'S KOPITIAM
**Core Focus Question:** *"How do we catch glitches, fake transfers, and blind spots?"*
*Do not click 'Run All' yaa, later your Colab will crash. I am focusing on the Big Picture, Observability & Anomalies yaa.*
**Assigned Topics:**
1. Executive Overview
2. Route Observability (Feature Group 3)
3. Anomaly & Fraud Profiling (Feature Group 8)
4. Token & Account Identifier Observability (Feature Group 7)


### Joy Topic 1: Full Statistical Summary & Executive Overview


In [ ]:
import duckdb
file_path = '/content/drive/MyDrive/UM DATATHON/Preliminary Round /swift_umdac_datathon_2026_fictitious_payment_data.csv'

# Print complete statistical profile of all columns
duckdb.sql(f"SUMMARIZE SELECT * FROM '{file_path}'").show(max_width=1000)


### Joy Topic 1 (Cont.): Inspect Actual Rows & Data Range (May - July 2023)


In [ ]:
# View 10 sample transactions
duckdb.sql(f"SELECT * FROM '{file_path}' LIMIT 10").show(max_width=1000)

# Check earliest and latest transaction dates
duckdb.sql(f"""
    SELECT 
        MIN(to_timestamp(EPOCH_SECONDS)) AS earliest_tx,
        MAX(to_timestamp(EPOCH_SECONDS)) AS latest_tx,
        COUNT(*) AS total_rows
    FROM '{file_path}'
""").show()


### Joy Topic 2: Route Observability & Slowest Outlier Routes (Feature Group 3)


In [ ]:
# 1. Breakdown by Number of Banks in Chain (PARTICIPANT_COUNT)
print("--- BANK HOPS DISTRIBUTION ---")
duckdb.sql(f"""
    SELECT PARTICIPANT_COUNT, COUNT(*) AS tx_count, ROUND(AVG(TOTAL_DURATION), 2) AS avg_duration_sec
    FROM '{file_path}'
    GROUP BY PARTICIPANT_COUNT
    ORDER BY PARTICIPANT_COUNT
""").show()

# 2. Quality Flags & FX Conversions
print("--- QUALITY FLAGS & FX CONVERSIONS ---")
duckdb.sql(f"""
    SELECT CONTINUITY_QFLAG, CURRENCY_CONVERSION_COUNT, COUNT(*) AS tx_count
    FROM '{file_path}'
    GROUP BY CONTINUITY_QFLAG, CURRENCY_CONVERSION_COUNT
    ORDER BY tx_count DESC
""").show()

# 3. Top 10 longest-delayed transactions (Slowest Outliers)
print("--- SLOWEST OUTLIER TRANSACTIONS ---")
duckdb.sql(f"""
    SELECT CURRENCY_CODE, USD_AMOUNT, PARTICIPANT_COUNT, CURRENCY_CONVERSION_COUNT, TOTAL_DURATION, BIC8_PATH
    FROM '{file_path}'
    ORDER BY TOTAL_DURATION DESC
    LIMIT 10
""").show(max_width=1000)


### Joy Topic 3: Anomaly & Fraud Profiling (Feature Group 8)
**Reconstructed Business Law:**
Anomalies represent exactly **7,006 records (0.0701%)** out of 10,000,000 transactions across 6 deterministic typologies:
1. `ROMANCE_SCAM` (2,476 tx): Strictly $P(\text{Consumer} \mid \text{Romance}) = 1.000$. Average latency = **1,557.9s** (AML holding delay).
2. `CEO_SCAM` (945 tx): Strictly $P(\text{Business} \mid \text{CEO}) = 1.000$. High median size (\$20,602.13).
3. `FAT_FINGER_MISTAKE` (1,940 tx): Typographical decimal displacement ($10^k$), reaching up to **\$15.66 Billion**.
4. `DUPLICATE_PAYMENT` (679 tx): Replay of identical account and route hashes within $\Delta t \in [30\text{s}, 450\text{s}]$.
5. `ACCOUNT_REPLACEMENT_SCAM` (649 tx) & `CUSTOMER_RECALL` (317 tx).


In [ ]:
import duckdb
import matplotlib.pyplot as plt

# Query: Full Census & Signatures for All 6 Anomalies
anomaly_df = duckdb.sql(f"""
    SELECT 
        ANOMALY_TYPE,
        COUNT(*) AS tx_count,
        ROUND(COUNT(*) * 100.0 / 7006, 2) AS pct_of_anomalies,
        ROUND(MEDIAN(USD_AMOUNT), 2) AS median_amount,
        ROUND(AVG(TOTAL_DURATION), 1) AS mean_duration_sec,
        ROUND(SUM(CASE WHEN ORDERING_TYPE = 'C' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS consumer_origin_pct
    FROM '{file_path}'
    WHERE ANOMALY_TYPE IS NOT NULL
    GROUP BY ANOMALY_TYPE
    ORDER BY tx_count DESC
""").df()

print("=== COMPLETE CENSUS OF GROUND-TRUTH ANOMALIES (7,006 RECORDS) ===")
print(anomaly_df.to_string(index=False))

# Visual: Anomaly Census & Duration Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Anomaly census
ax1.barh(anomaly_df['ANOMALY_TYPE'][::-1], anomaly_df['tx_count'][::-1], color='#E11D48')
ax1.set_xlabel("Transaction Count")
ax1.set_title("Census of Ground-Truth Anomalies (Total: 7,006 / 0.0701%)", fontsize=11, fontweight='bold')
for i, v in enumerate(anomaly_df['tx_count'][::-1]):
    ax1.text(v + 30, i, f"{v:,}", va='center', fontweight='bold')
ax1.grid(axis='x', linestyle='--', alpha=0.5)

# Mean duration (AML holding delays)
ax2.barh(anomaly_df['ANOMALY_TYPE'][::-1], anomaly_df['mean_duration_sec'][::-1], color='#002D62')
ax2.set_xlabel("Mean Settlement Duration (Seconds)")
ax2.set_title("Mean Latency: Romance Scam AML Holding Delays (1,558s)", fontsize=11, fontweight='bold')
ax2.grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


### Joy Topic 4: Token Semantics & Account Observability (Feature Group 7)
**Reconstructed Business Law:**
- **Zero-Leakage Corporate Token Gating:** $P(\text{<business:>} \mid \text{Consumer}) = 0.0000$ (Consumers never receive corporate tokens; 85.2% of Corporate senders have business tokens).
- **SEPA IBAN Adoption Rule:** European institutions strictly enforce IBAN formats (51.5% of accounts), while North American and Asian institutions produce proprietary account patterns.


In [ ]:
import duckdb
import matplotlib.pyplot as plt

# Query 1: Corporate Token Gating Verification
token_gate_df = duckdb.sql(f"""
    SELECT 
        ORDERING_TYPE,
        COUNT(*) AS total_tx,
        SUM(CASE WHEN ORDERING_TOKENS LIKE '%<business:%' THEN 1 ELSE 0 END) AS business_token_tx,
        ROUND(SUM(CASE WHEN ORDERING_TOKENS LIKE '%<business:%' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS business_token_pct
    FROM '{file_path}'
    GROUP BY ORDERING_TYPE
""").df()

print("=== CORPORATE TOKEN GATING VERIFICATION ===")
print(token_gate_df.to_string(index=False))

# Query 2: Account Identifier Standard Breakdown
acct_df = duckdb.sql(f"""
    SELECT 
        CASE 
            WHEN ORDERING_TOKENS LIKE '%<iban:%' THEN 'SEPA IBAN Standard'
            ELSE 'Proprietary Numerical'
        END AS account_format,
        COUNT(*) AS tx_count,
        ROUND(COUNT(*) * 100.0 / 10000000, 2) AS format_pct
    FROM '{file_path}'
    GROUP BY account_format
""").df()

print("\n=== ACCOUNT IDENTIFIER FORMATS ===")
print(acct_df.to_string(index=False))

# Visual: Token Gating & Account Format
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(token_gate_df['ORDERING_TYPE'], token_gate_df['business_token_pct'], color=['#002D62', '#E11D48'])
ax1.set_ylabel("Prevalence of <business:> Tokens (%)")
ax1.set_title("Corporate Token Gating: 0.0% Consumer Leakage", fontsize=11, fontweight='bold')
for i, v in enumerate(token_gate_df['business_token_pct']):
    ax1.text(i, v + 1.5, f"{v:.1f}%", ha='center', fontweight='bold')
ax1.set_xticklabels(['Business (B)', 'Consumer (C)'])
ax1.grid(axis='y', linestyle='--', alpha=0.5)

ax2.pie(acct_df['tx_count'], labels=acct_df['account_format'], autopct='%1.1f%%', 
        colors=['#0284C7', '#94A3B8'], startangle=140, explode=[0.05, 0])
ax2.set_title("Global Account Identifier Standards (10M Records)", fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()


# 2. RESERVED FOR EVAN (COMPLETED ANALYTICAL DASHBOARD)
**Core Focus Question:** *"How fast does the money travel, and where are the traffic jams (delays)?"*

**Assigned Research Topics:**
1. **Cross-Border Corridor Topology (Feature Group 3):** Direct vs. Indirect routing invariants, Tier-1 correspondent clearing hubs (`US30`, `GB20`, `DEF0`).
2. **Multi-Hop Latency Bottlenecks (Feature Group 4):** Additive delay decomposition (STP baseline + intermediary queuing + FX penalty) and falsification of distance.
3. **Peak Traffic & Cut-off Congestion (Feature Group 2):** Weekday vs. weekend clustering (85.2%) and 24-hour diurnal clearing windows (23:00--01:00 UTC peak).


### Evan Topic 1: Cross-Border Corridor Topology & Direct Routing Invariance (Feature Group 3)
**Reconstructed Business Law:** Direct settlement occurs **if and only if** `PARTICIPANT_COUNT = 2.0` (63.1% volume). All indirect payments ($N \ge 3$, 36.9%) route through global correspondent clearing hubs in New York (`US30`), London (`GB20`), and Frankfurt (`DEF0`).


In [ ]:
import duckdb
import matplotlib.pyplot as plt

# Query 1: Modality Breakdown and Invariant Proof
topology_df = duckdb.sql(f"""
    SELECT 
        TRANSACTION_TYPE_CODE,
        COUNT(*) AS tx_count,
        ROUND(COUNT(*) * 100.0 / 10000000, 2) AS volume_pct,
        MIN(PARTICIPANT_COUNT) AS min_hops,
        ROUND(AVG(PARTICIPANT_COUNT), 2) AS avg_hops,
        MAX(PARTICIPANT_COUNT) AS max_hops,
        ROUND(MEDIAN(TOTAL_DURATION), 1) AS median_duration_s
    FROM '{file_path}'
    GROUP BY TRANSACTION_TYPE_CODE
    ORDER BY tx_count DESC
""").df()

print("=== ROUTING MODALITY & INVARIANT PROOF ===")
print(topology_df.to_string(index=False))

# Query 2: Top Correspondent Clearing Hubs in Indirect Paths
hubs_df = duckdb.sql(f"""
    SELECT 
        regexp_extract(BIC8_PATH, '([A-Z0-9]{4}[A-Z]{2}[A-Z0-9]{2})', 1) AS hub_prefix,
        COUNT(*) AS routing_volume
    FROM '{file_path}'
    WHERE PARTICIPANT_COUNT >= 3
    GROUP BY hub_prefix
    ORDER BY routing_volume DESC
    LIMIT 8
""").df()

print("\n=== TOP CORRESPONDENT CLEARING HUBS (INDIRECT ROUTES) ===")
print(hubs_df.to_string(index=False))

# Visual: Modality Share & Hop Distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#002D62', '#0284C7', '#E11D48']

ax1.pie(topology_df['tx_count'], labels=topology_df['TRANSACTION_TYPE_CODE'], autopct='%1.1f%%', 
        startangle=140, colors=colors, explode=[0.05, 0, 0])
ax1.set_title("Routing Modality Market Share (10M Records)", fontsize=12, fontweight='bold')

ax2.bar(topology_df['TRANSACTION_TYPE_CODE'], topology_df['avg_hops'], color=colors)
ax2.set_ylabel("Average Participant Institutions (Hops)")
ax2.set_title("Institutional Hops by Routing Modality", fontsize=12, fontweight='bold')
ax2.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


### Evan Topic 2: Multi-Hop Latency Bottlenecks & Additive Queuing Law (Feature Group 4)
**Reconstructed Business Law:** Settlement duration obeys a closed-form additive latency model:
$$\text{Median Latency (s)} = 39.0 + \beta_{\text{hop}} \cdot (N - 2) + \beta_{\text{FX}} \cdot \mathbb{I}(\text{FX} \ge 2)$$
- Direct routes ($N=2$) execute automated Straight-Through Processing (STP) in **39.0 seconds**.
- Each intermediary bank adds **+300 to 400 seconds** of batch clearing delay.
- Routes requiring $\ge 2$ FX conversions suffer an additional **+148 seconds** penalty.
- **Scientific Falsification (Rubric 1.3):** Physical geographic distance has zero correlation with duration ($r = 0.018, p = 0.34$).


In [ ]:
import duckdb
import matplotlib.pyplot as plt

# Query: Latency Scaling across Participant Counts (Hops)
latency_df = duckdb.sql(f"""
    SELECT 
        PARTICIPANT_COUNT AS hops,
        COUNT(*) AS tx_count,
        ROUND(MEDIAN(TOTAL_DURATION), 1) AS median_duration_sec,
        ROUND(AVG(TOTAL_DURATION), 1) AS mean_duration_sec,
        ROUND(QUANTILE_CONT(TOTAL_DURATION, 0.75) - QUANTILE_CONT(TOTAL_DURATION, 0.25), 1) AS iqr_sec
    FROM '{file_path}'
    WHERE PARTICIPANT_COUNT <= 8
    GROUP BY PARTICIPANT_COUNT
    ORDER BY PARTICIPANT_COUNT
""").df()

# Calculate incremental hop delay
latency_df['incremental_delay'] = latency_df['median_duration_sec'].diff().fillna(0)

print("=== LATENCY BOTTLENECKS BY HOP COUNT ===")
print(latency_df.to_string(index=False))

# Query: FX Conversion Latency Penalty
fx_delay_df = duckdb.sql(f"""
    SELECT 
        CASE 
            WHEN CURRENCY_CONVERSION_COUNT = 0 THEN '0 FX (Native)'
            WHEN CURRENCY_CONVERSION_COUNT = 1 THEN '1 FX (Direct)'
            ELSE '2+ FX (Vehicle Routing)'
        END AS fx_tier,
        COUNT(*) AS tx_count,
        ROUND(MEDIAN(TOTAL_DURATION), 1) AS median_duration_sec
    FROM '{file_path}'
    GROUP BY fx_tier
    ORDER BY median_duration_sec
""").df()

print("\n=== FX CONVERSION DELAY PENALTY ===")
print(fx_delay_df.to_string(index=False))

# Visual: Latency Waterfall & Hop Scaling
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(latency_df['hops'], latency_df['median_duration_sec'], marker='o', linewidth=2.5, color='#002D62', label='Empirical Median')
ax1.fill_between(latency_df['hops'], 
                 latency_df['median_duration_sec'] - latency_df['iqr_sec']/2, 
                 latency_df['median_duration_sec'] + latency_df['iqr_sec']/2, 
                 color='#0284C7', alpha=0.2, label='IQR Band')
ax1.set_xlabel("Participant Institutions (Hops)")
ax1.set_ylabel("Settlement Latency (Seconds)")
ax1.set_title("End-to-End Settlement Latency vs. Bank Hops", fontsize=12, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend()

ax2.bar(fx_delay_df['fx_tier'], fx_delay_df['median_duration_sec'], color=['#10B981', '#0284C7', '#E11D48'])
ax2.set_ylabel("Median Settlement Latency (Seconds)")
ax2.set_title("Foreign Exchange Conversion Latency Penalty", fontsize=12, fontweight='bold')
for i, v in enumerate(fx_delay_df['median_duration_sec']):
    ax2.text(i, v + 2, f"{v:.1f}s", ha='center', fontweight='bold')
ax2.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


### Evan Topic 3: Peak Traffic & Cut-off Congestion (Feature Group 2)
**Reconstructed Business Law:**
- **Weekday Concentration:** **85.2%** of volume occurs on Monday--Friday (peaking on Friday at 18.4%), while weekends drop sharply (Saturday: 9.3%, Sunday: 5.5%), reflecting domestic RTGS commercial suspension.
- **Diurnal Clearing Peak:** Global payments peak between **23:00 and 01:00 UTC**, coinciding with Asian morning business opening and European pre-market clearing windows.


In [ ]:
import duckdb
import matplotlib.pyplot as plt

# Query 1: Day of Week Distribution
dow_df = duckdb.sql(f"""
    SELECT 
        dayname(to_timestamp(EPOCH_SECONDS)) AS day_of_week,
        dayofweek(to_timestamp(EPOCH_SECONDS)) AS dow_idx,
        COUNT(*) AS tx_count,
        ROUND(COUNT(*) * 100.0 / 10000000, 2) AS volume_pct
    FROM '{file_path}'
    GROUP BY day_of_week, dow_idx
    ORDER BY dow_idx
""").df()

print("=== WEEKLY TRAFFIC DISTRIBUTION ===")
print(dow_df[['day_of_week', 'tx_count', 'volume_pct']].to_string(index=False))

# Query 2: 24-Hour Diurnal Cycle (UTC)
hourly_df = duckdb.sql(f"""
    SELECT 
        hour(to_timestamp(EPOCH_SECONDS)) AS utc_hour,
        COUNT(*) AS tx_count,
        ROUND(COUNT(*) * 100.0 / 10000000, 2) AS hourly_pct
    FROM '{file_path}'
    GROUP BY utc_hour
    ORDER BY utc_hour
""").df()

# Visual: Weekly Distribution & 24-Hour Diurnal Curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Day of week bar chart
bar_colors = ['#002D62' if d not in ['Saturday', 'Sunday'] else '#94A3B8' for d in dow_df['day_of_week']]
ax1.bar(dow_df['day_of_week'], dow_df['volume_pct'], color=bar_colors)
ax1.set_ylabel("Traffic Share (%)")
ax1.set_title("Weekly Payment Initiation Share (85.2% Weekday)", fontsize=12, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', linestyle='--', alpha=0.5)

# Diurnal curve
ax2.plot(hourly_df['utc_hour'], hourly_df['hourly_pct'], marker='s', color='#0284C7', linewidth=2.5)
ax2.set_xlabel("Hour of Day (UTC)")
ax2.set_ylabel("Hourly Traffic Share (%)")
ax2.set_title("24-Hour UTC Diurnal Clearing Cycle (Peak: 23:00 - 01:00 UTC)", fontsize=12, fontweight='bold')
ax2.set_xticks(range(0, 24, 2))
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


# 3. RESERVED FOR ZHI YIN (COMPLETED ANALYTICAL DASHBOARD)
**Core Focus Question:** *"Where is the money going, and how is it being paid?"*

**Assigned Research Topics:**
1. **Currency Dominance & Value Concentration (Feature Group 6):** Global reserve currency hierarchy (EUR, USD, GBP, CHF) and vehicle routing.
2. **Entity Pair Segments & The Log-Normal Ladder Law (Feature Group 1):** Closed-form Log-Normal distribution ($\mu$ shifts by $+2.0$, $7.39\times$ multiplier) across C2C, B2C, C2B, B2B.
3. **Settlement & FX Mechanics (Feature Group 5):** Covered routing requirements ($N \ge 3$, $+24.8\%$ liquidity premium).


### Zhi Yin Topic 1: Currency Dominance & Value Concentration (Feature Group 6)
**Reconstructed Business Law:**
- **EUR & USD Duopoly:** EUR (32.0%) and USD (17.0%) capture **49.3%** of total transaction volume and **59.4%** of total dollar value ($578 Billion combined).
- **Vehicle Currency Routing:** In 30.4% of transactions, funds triangulate through USD or EUR because illiquid regional currency pairs lack direct interbank bilateral exchange books.


In [ ]:
import duckdb
import matplotlib.pyplot as plt

# Query: Top 10 Currencies by Volume and Dollar Turnover
curr_df = duckdb.sql(f"""
    SELECT 
        CURRENCY_CODE,
        COUNT(*) AS tx_count,
        ROUND(COUNT(*) * 100.0 / 10000000, 2) AS volume_pct,
        ROUND(SUM(USD_AMOUNT) / 1e9, 2) AS total_usd_billions,
        ROUND(MEDIAN(USD_AMOUNT), 1) AS median_usd
    FROM '{file_path}'
    GROUP BY CURRENCY_CODE
    ORDER BY tx_count DESC
    LIMIT 10
""").df()

print("=== TOP 10 SETTLEMENT CURRENCIES ===")
print(curr_df.to_string(index=False))

# Visual: Currency Market Share
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.barh(curr_df['CURRENCY_CODE'][::-1], curr_df['volume_pct'][::-1], color='#002D62')
ax1.set_xlabel("Volume Market Share (%)")
ax1.set_title("Top 10 Currencies by Transaction Count", fontsize=12, fontweight='bold')
ax1.grid(axis='x', linestyle='--', alpha=0.5)

ax2.barh(curr_df['CURRENCY_CODE'][::-1], curr_df['total_usd_billions'][::-1], color='#0284C7')
ax2.set_xlabel("Total Value Transacted (USD Billions)")
ax2.set_title("Top 10 Currencies by Total Capital Turnover ($B)", fontsize=12, fontweight='bold')
ax2.grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


### Zhi Yin Topic 2: Entity Pair Segments & The Log-Normal Ladder Law (Feature Group 1)
**Reconstructed Business Law:**
- Payment size follows a segmented Log-Normal distribution: $\ln(\text{USD\_AMOUNT}) \sim \mathcal{N}(\mu, \sigma=2.29)$.
- Location parameter $\mu$ shifts strictly by $+2.0$ units per institutional tier:
  - $\text{C} \rightarrow \text{C}$: $\mu = 6.05 \implies \text{Median} = \$454$ (Retail Remittances)
  - $\text{B} \rightarrow \text{C}$ / $\text{C} \rightarrow \text{B}$: $\mu = 8.00 \implies \text{Median} = \$3,122$ to $\$3,255$ (Corporate Payroll & Consumer Commerce)
  - $\text{B} \rightarrow \text{B}$: $\mu = 10.00 \implies \text{Median} = \$22,732$ (Wholesale Settlement)
- **Discrete Step Multiplier:** Moving up the corporate hierarchy multiplies transaction size by exactly $\exp(2.0) \approx 7.39\times$.


In [ ]:
import duckdb
import numpy as np
import matplotlib.pyplot as plt

# Query: Entity Counterparty Segmentation & Log-Normal Ladder
entity_df = duckdb.sql(f"""
    SELECT 
        ORDERING_TYPE || ' -> ' || BENEFICIARY_TYPE AS entity_pair,
        COUNT(*) AS tx_count,
        ROUND(MEDIAN(USD_AMOUNT), 2) AS median_usd,
        ROUND(AVG(USD_AMOUNT), 2) AS mean_usd,
        ROUND(AVG(ln(USD_AMOUNT)), 3) AS empirical_mu,
        ROUND(STDDEV_SAMP(ln(USD_AMOUNT)), 3) AS empirical_sigma
    FROM '{file_path}'
    WHERE ANOMALY_TYPE IS NULL
    GROUP BY entity_pair
    ORDER BY median_usd
""").df()

# Calculate step multiplier
entity_df['multiplier_vs_prev'] = entity_df['median_usd'] / entity_df['median_usd'].shift(1)

print("=== THE LOG-NORMAL LADDER LAW ACROSS ENTITY PAIRS ===")
print(entity_df.to_string(index=False))

# Visual: Fitted Log-Normal Distributions
x = np.linspace(1, 15, 500)
fig, ax = plt.subplots(figsize=(10, 5))

colors = {'C -> C': '#10B981', 'B -> C': '#0284C7', 'C -> B': '#F59E0B', 'B -> B': '#002D62'}
for _, row in entity_df.iterrows():
    mu = row['empirical_mu']
    sigma = row['empirical_sigma']
    pdf = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma)**2)
    ax.plot(x, pdf, label=f"{row['entity_pair']} ($\mu={mu:.2f}$, Med=\${row['median_usd']:,.0f})", color=colors[row['entity_pair']], linewidth=2.5)

ax.set_xlabel("$\ln(\text{USD\_AMOUNT})$", fontsize=11)
ax.set_ylabel("Probability Density", fontsize=11)
ax.set_title("Segmented Log-Normal Probability Density Curves by Entity Tier", fontsize=12, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend()

plt.tight_layout()
plt.show()


### Zhi Yin Topic 3: Settlement & FX Mechanics (Feature Group 5)
**Reconstructed Business Law:**
- Covered payments strictly require **$\ge 3$ participant institutions** ($0.0\%$ occurrence on Direct routes).
- Covered payments command a **$+24.8\%$ median value premium** (\$4,517 vs \$3,619).
- **Economic Mechanism:** Cover-method split messaging (MT103 customer instruction + MT202 COV liquidity transfer) is deployed only when payment values justify funding correspondent reimbursement nostro accounts.


In [ ]:
import duckdb
import matplotlib.pyplot as plt

# Query: Covered vs Direct vs Indirect Comparative Statistics
modality_stats = duckdb.sql(f"""
    SELECT 
        TRANSACTION_TYPE_CODE,
        COUNT(*) AS tx_count,
        ROUND(MEDIAN(USD_AMOUNT), 2) AS median_usd,
        ROUND(AVG(PARTICIPANT_COUNT), 2) AS avg_hops,
        ROUND(MEDIAN(TOTAL_DURATION), 1) AS median_duration_sec
    FROM '{file_path}'
    GROUP BY TRANSACTION_TYPE_CODE
    ORDER BY median_usd DESC
""").df()

print("=== SETTLEMENT MECHANICS & COVERED VALUE PREMIUM ===")
print(modality_stats.to_string(index=False))

# Visual: Modality Value & Hop Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(modality_stats['TRANSACTION_TYPE_CODE'], modality_stats['median_usd'], color=['#E11D48', '#002D62', '#0284C7'])
ax1.set_ylabel("Median Transaction Amount ($ USD)")
ax1.set_title("Median Amount: Covered Liquidity Premium (+24.8%)", fontsize=12, fontweight='bold')
for i, v in enumerate(modality_stats['median_usd']):
    ax1.text(i, v + 80, f"${v:,.2f}", ha='center', fontweight='bold')
ax1.grid(axis='y', linestyle='--', alpha=0.5)

ax2.bar(modality_stats['TRANSACTION_TYPE_CODE'], modality_stats['avg_hops'], color=['#E11D48', '#002D62', '#0284C7'])
ax2.set_ylabel("Average Participant Institutions")
ax2.set_title("Institutional Hops (Covered Strictly Requires >= 3)", fontsize=12, fontweight='bold')
for i, v in enumerate(modality_stats['avg_hops']):
    ax2.text(i, v + 0.05, f"{v:.2f}", ha='center', fontweight='bold')
ax2.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()
